In [6]:
from catboost import CatBoostClassifier
from catboost import Pool
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [7]:
seed = 42

In [8]:
df = pd.read_csv("Data/data_kern_ratios_06_10.csv")

In [9]:
elements = ['Mg', 'Al', 'Si', 'P', 'S', 'K', 'Ca', 'Ti', 'Mn', 'Fe',
    'Ni', 'Cu', 'Zn', 'Se', 'Rb', 'Sr', 'Zr', 'Mo']

In [10]:
df[elements]

,Mg,Al,Si,P,S,K,Ca,Ti,Mn,Fe,Ni,Cu,Zn,Se,Rb,Sr,Zr,Mo
0,1.821148,7.515664,18.611483,0.299690,10.465318,0.942543,1.469765,0.070000,0.023969,5.472308,0.003111,0.005070,0.008353,0.000145,0.004428,0.017295,0.005169,0.006378
1,1.266357,7.461183,16.904798,0.000239,14.653735,0.883390,0.000585,0.140444,0.008495,6.964299,0.002921,0.006038,0.012220,0.000321,0.003050,0.009028,0.003092,0.007169
2,1.724880,7.150965,18.068731,0.000239,13.442425,0.930935,0.054942,0.070000,0.022970,5.942331,0.001074,0.005473,0.008705,0.000180,0.003570,0.018256,0.005030,0.006039
3,1.646439,8.260792,21.260196,0.000239,10.257598,1.235161,0.000585,0.184038,0.014205,4.126949,0.003727,0.005191,0.008845,0.000180,0.004764,0.011463,0.004061,0.005446
4,1.847176,7.757235,19.837340,0.000239,10.623499,1.059210,0.478515,0.070000,0.022656,5.052294,0.003111,0.007288,0.013555,0.000286,0.004397,0.016122,0.002446,0.006519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3023,2.129802,4.410461,8.497462,1.992644,10.053492,0.514109,4.325427,0.070000,0.107449,13.178634,0.000979,0.001605,0.006033,0.000003,0.001672,0.029409,0.003277,0.000442
3024,1.863934,12.555305,24.545034,0.084607,1.785869,1.701369,0.058194,0.397704,0.020486,2.164767,0.000553,0.001605,0.008986,0.000003,0.005071,0.011373,0.008861,0.000442
3025,1.705032,12.769895,24.812063,0.020000,1.558093,1.863352,0.000585,0.397608,0.020115,2.041525,0.000790,0.001605,0.008634,0.000003,0.005346,0.011643,0.008630,0.000442
3026,1.680073,12.810209,25.421817,0.038680,1.082697,1.895027,0.000585,0.408698,0.014690,1.656151,0.000600,0.001605,0.008213,0.000003,0.005622,0.012064,0.009830,0.000442


In [12]:
def create_synthetic_slag(df, elements, window_m, noise_level=0, seed=42):
    """
    Создаёт синтетический шлам из керна.
    
    Параметры:
    ----------
    df : pd.DataFrame
        Датасет с керном. Колонки: Well, Depth, Strat, elements
    elements : list
        Список элементов (18 штук)
    window_m : float
        Ширина окна перемешивания в метрах
    noise_level : float
        Уровень шума: 0 (без шума), 0.1, 0.3, 0.5
        σ = noise_level × std(element)
    seed : int
        Random seed для воспроизводимости
    
    Возвращает:
    -----------
    df_slag : pd.DataFrame
        Колонки: Well, Depth, Strat, {el}_kern, {el}_clean, {el}_slag
    """
    np.random.seed(seed)
    
    sigma_m = window_m / 3
    df_slag_list = []
    
    for well in df['Well'].unique():
        df_well = df[df['Well'] == well].sort_values('Depth').reset_index(drop=True)
        
        depths = df_well['Depth'].values
        n = len(depths)
        
        # std ПО ЭТОЙ СКВАЖИНЕ
        std_per_el = df_well[elements].std()
        
        df_well_slag = df_well[['Well', 'Depth', 'Strat']].copy()
        
        for el in elements:
            values = df_well[el].values
            slag_clean = np.zeros(n)
            
            for j in range(n):
                d_center = depths[j]
                mask = (depths >= d_center - window_m) & (depths <= d_center)
                d_window = depths[mask]
                v_window = values[mask]
                
                distances = d_center - d_window
                weights = np.exp(-0.5 * (distances / sigma_m) ** 2)
                weights = weights / weights.sum()
                
                slag_clean[j] = np.sum(v_window * weights)
            
            # Шум — пропорционален std ЭТОЙ скважины
            if noise_level > 0:
                sigma_noise = noise_level * std_per_el[el]
                noise = np.random.normal(0, sigma_noise, size=n)
                slag_noisy = slag_clean + noise
            else:
                slag_noisy = slag_clean.copy()
            
            df_well_slag[f'{el}_kern'] = values
            df_well_slag[f'{el}_clean'] = slag_clean
            df_well_slag[f'{el}_slag'] = slag_noisy
        
        df_slag_list.append(df_well_slag)
    
    df_slag = pd.concat(df_slag_list, ignore_index=True)
    return df_slag

In [13]:
windows = [0.5, 1.0, 2.0, 3.0]
noise_levels = [0.1, 0.3, 0.5]

for window_m in windows:
    # Без шума
    df_slag = create_synthetic_slag(df, elements, window_m, noise_level=0)
    df_slag.to_csv(f'Result/Synthetic_slag/synthetic_slag_w{window_m}_no_noise.csv', index=False)
    
    # С шумом
    for level in noise_levels:
        df_slag = create_synthetic_slag(df, elements, window_m, noise_level=level)
        df_slag.to_csv(f'Result/Synthetic_slag/synthetic_slag_w{window_m}_s{level}.csv', index=False)